In [56]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from scipy.stats import randint, uniform

In [57]:
df = pd.read_parquet('cleaned_eta.parquet', engine='pyarrow')
df.head()

,order_id,store_id,zone,store_name,order_timestamp,total_items,fresh_item_count,picking_capacity,rider_capacity,traffic_level,weather_condition,actual_delivery_minutes,promised_delivery_mintes,distance_km,hour,is_peak_hour,day_name,is_weekend
0,ord_000001,str_000032,HSR Extension,QuickMart Whitefield Main Road,2026-01-01 00:03:00,2,0,10,12,Normal,Clear,11.0,14.0,1.73,0,False,Thursday,False
1,ord_000002,str_000029,BTM Layout,QuickMart HSR Extension,2026-01-01 00:07:00,4,0,13,12,Normal,Clear,11.0,12.0,1.11,0,False,Thursday,False
2,ord_000003,str_000001,Malleshwaram,QuickMart Koramangala,2026-01-01 00:11:00,2,1,14,17,Normal,Clear,11.0,13.0,1.51,0,False,Thursday,False
3,ord_000004,str_000038,Kalyan Nagar,QuickMart Nagarbhavi,2026-01-01 00:11:00,3,0,21,20,Normal,Clear,9.0,11.0,1.15,0,False,Thursday,False
4,ord_000005,str_000011,Jayanagar,QuickMart Hebbal,2026-01-01 00:12:00,6,2,8,11,High,Clear,13.0,15.0,1.01,0,False,Thursday,False


In [58]:
df["is_peak_hour"] = df["is_peak_hour"].astype(int)
df["is_weekend"] = df["is_weekend"].astype(int)

In [59]:
df["high_risk_condition"] = (
    (df["traffic_level"] == "High") & (df["weather_condition"] == "Heavy Rain")
).astype(int)

In [60]:
df[["high_risk_condition", "actual_delivery_minutes"]].corr()

,high_risk_condition,actual_delivery_minutes
high_risk_condition,1.000000,0.096513
actual_delivery_minutes,0.096513,1.000000


In [61]:
df.head()

,order_id,store_id,zone,store_name,order_timestamp,total_items,fresh_item_count,picking_capacity,rider_capacity,traffic_level,weather_condition,actual_delivery_minutes,promised_delivery_mintes,distance_km,hour,is_peak_hour,day_name,is_weekend,high_risk_condition
0,ord_000001,str_000032,HSR Extension,QuickMart Whitefield Main Road,2026-01-01 00:03:00,2,0,10,12,Normal,Clear,11.0,14.0,1.73,0,0,Thursday,0,0
1,ord_000002,str_000029,BTM Layout,QuickMart HSR Extension,2026-01-01 00:07:00,4,0,13,12,Normal,Clear,11.0,12.0,1.11,0,0,Thursday,0,0
2,ord_000003,str_000001,Malleshwaram,QuickMart Koramangala,2026-01-01 00:11:00,2,1,14,17,Normal,Clear,11.0,13.0,1.51,0,0,Thursday,0,0
3,ord_000004,str_000038,Kalyan Nagar,QuickMart Nagarbhavi,2026-01-01 00:11:00,3,0,21,20,Normal,Clear,9.0,11.0,1.15,0,0,Thursday,0,0
4,ord_000005,str_000011,Jayanagar,QuickMart Hebbal,2026-01-01 00:12:00,6,2,8,11,High,Clear,13.0,15.0,1.01,0,0,Thursday,0,0


In [62]:
traffic_map = {"Normal": 1, "High": 2, "Very High": 3}
df["traffic_level"] = df["traffic_level"].map(traffic_map)

In [63]:
df[["traffic_level", "actual_delivery_minutes"]].corr()

,traffic_level,actual_delivery_minutes
traffic_level,1.000000,0.401549
actual_delivery_minutes,0.401549,1.000000


In [64]:
features = [
    "distance_km",
    "hour",
    "store_name",
    "total_items",
    "fresh_item_count",
    "is_peak_hour",
    "traffic_level",
    "weather_condition",
    "high_risk_condition"
]

In [65]:
target = ['actual_delivery_minutes']

In [66]:
numeric_features = ["distance_km", "fresh_item_count", "hour", "total_items", 'traffic_level']

categorical_features = ["store_name" ,"weather_condition"]

binary_features = ["is_peak_hour",  "high_risk_condition"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="passthrough",
)

In [67]:
X = df[features].copy()
y = df[target].copy()

In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [69]:
y_pred_baseline = [y_train.mean()] * len(y_test)

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)

baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE  : {baseline_mae:.2f} minutes")
print(f"Baseline RMSE : {baseline_rmse:.2f} minutes")
print(f"Baseline R²   : {baseline_r2:.3f}")

Baseline MAE  : 3.38 minutes
Baseline RMSE : 4.28 minutes
Baseline R²   : -0.000


In [70]:
linear_model = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", LinearRegression())]
)

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

In [71]:
# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Display results
print("Linear Regression Results")
print("-" * 30)
print(f"MAE  : {mae:.2f} minutes")
print(f"RMSE : {rmse:.2f} minutes")
print(f"R²   : {r2:.3f}")

Linear Regression Results
------------------------------
MAE  : 1.56 minutes
RMSE : 1.97 minutes
R²   : 0.788


In [72]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
xgb_pred = xgb_model.predict(X_test)

In [73]:
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Results")
print("-" * 30)
print(f"MAE  : {xgb_mae:.2f} minutes")
print(f"RMSE : {xgb_rmse:.2f} minutes")
print(f"R²   : {xgb_r2:.3f}")

XGBoost Results
------------------------------
MAE  : 1.42 minutes
RMSE : 1.80 minutes
R²   : 0.823


In [74]:
import joblib

joblib.dump(xgb_model, "final_eta_model.pkl")

print("Model saved successfully.")

Model saved successfully.
